### Feature Selection using HoldOut

Following initial data preprocessing steps, which included the removal of several irrelevant columns, we employed Recursive Feature Elimination (RFE) to determine the optimal feature subset for a linear regression model. 

A holdout validation approach was utilized to evaluate model performance with varying numbers of features, using the Mean Absolute Error (MAE) as the primary evaluation metric.

In [ ]:
#Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=44
)

In [ ]:
grid = np.arange(1,n_total+1)

In [ ]:
# Create variables to store the records for training and validation
records = []
best_model = None
high_score = None
n_feat = 0
selected_features = None

In [ ]:
for n_sel in grid:
    model = LinearRegression()
    pipe = Pipeline([
        ("prep", pipe_preprocessor),
        ("rfe", RFE(estimator=model, n_features_to_select=n_sel)),
        ("linreg", LinearRegression()),
    ])
    
    pipe.fit(X_train, y_train) 

    # results of training dataset
    train_pred = pipe.predict(X_train)
    train_mae = mean_absolute_error(y_train, train_pred)

    # results of validation dataset
    val_pred = pipe.predict(X_val)
    val_mae = mean_absolute_error(y_val, val_pred)

    # append both records to then plot
    records.append({"n_selected": n_sel,
    "train_MAE": train_mae,
    "val_MAE": val_mae})

    if (high_score is None) or (val_mae < high_score):
        high_score = val_mae
        best_model = pipe
        n_feat = n_sel

In [ ]:
# Checking the scores for each iteration
records

In [ ]:
# extract to list to then plot
train_score_list = [record["train_MAE"] for record in records]
val_score_list = [record["val_MAE"] for record in records]

len(val_score_list)

In [ ]:
plt.plot(list(range(1,66)), train_score_list, label="Score on Training Set", color='yellowgreen')
plt.plot(list(range(1,66)), val_score_list, label="Score on Validation Set", color='dimgray')
plt.xlabel("Number of features")
plt.ylabel("Score")
plt.legend()
plt.show()

print("Lowest score of MAE is with number of features: %d" %n_feat)
print("Score with %d features: %f" % (n_feat, high_score))

- Altough the lowest score is with 62 features, we can say that using the Linear Regression with the RFE, the model reaches his plateu at 57 features.
- The model does not seem to be overfitting neither underfitting, since  the difference on training and validation score is very low.
- The MAE got lower as we increased the number of features, however the error is still high with 2.8k, wich may indicate that the relathionship between X and Y is not that linear.